# `big-code-analysis` Python bindings — Jupyter quick start

This notebook walks through the canonical data-science workflow: install the wheel, analyse a small repository, materialise the results as a `pandas.DataFrame`, plot per-function complexity, and surface the top-N most complex functions. The same notebook is executed by CI via `jupyter nbconvert --execute --inplace` so the cells below stay in lockstep with the latest API.

The repository under analysis is the bindings crate's own test fixture tree — small, deterministic, multi-language, and already available on every check-out. Set `BCA_NOTEBOOK_FIXTURES` to override the path, or edit the `FIXTURES` line below to point at any directory of source files.

## 1. Install the wheel

The cell below is commented because both the notebook tests in CI and most contributors will already have `big_code_analysis` installed via `maturin develop`. Uncomment and run in a fresh kernel.

In [ ]:
# %pip install big-code-analysis jupyter pandas matplotlib

## 2. Imports and fixture location

`big_code_analysis` is the public Python package; `bca` is the conventional alias and the one used throughout the project's book.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import big_code_analysis as bca
import matplotlib.pyplot as plt
import pandas as pd

# Fixture resolution: env var FIRST (the contract the CI step and
# the test harness both use), then a notebook-relative fallback.
# The fallback uses Path.cwd() because the notebook has no __file__
# under nbconvert; nbconvert's `--execute` sets the kernel cwd to
# the notebook's parent directory, so Path.cwd() == examples/ on
# the supported invocation. A copy-pasted notebook run from any
# other cwd is a documented foot-gun — set BCA_NOTEBOOK_FIXTURES
# in that case.
_env = os.environ.get("BCA_NOTEBOOK_FIXTURES")
FIXTURES = Path(_env) if _env else (Path.cwd().parent / "tests" / "fixtures")

# Positive sanity check on a known-present file, NOT just
# `is_dir()`. The workspace's other `tests/fixtures/` (the Rust
# integration submodule) also exists and would pass `is_dir()`
# while containing zero `.rs` source files — silently producing
# an empty DataFrame later. `hello.rs` is the canonical fixture
# and present in every check-out.
assert (FIXTURES / "hello.rs").is_file(), (
    f"fixtures dir does not contain hello.rs: {FIXTURES}. "
    "Set BCA_NOTEBOOK_FIXTURES to your fixture root, or run the "
    "notebook from big-code-analysis-py/examples/."
)
print(f"Analysing fixtures under: {FIXTURES}")

## 3. Discover source files

Use `bca.language_for_file(path)` to identify source files: the helper matches the path extension and falls back to `#!` shebang / emacs `-*- mode -*-` detection so extension-less scripts (like the bindings' `install` fixture) are included. `rglob` walks recursively so subdirectories are covered too.

In [ ]:
sources: list[Path] = []
for candidate in sorted(FIXTURES.rglob("*")):
    if not candidate.is_file():
        continue
    try:
        lang = bca.language_for_file(candidate)
    except OSError:
        # Permission errors / dangling symlinks: just skip.
        continue
    if lang is not None:
        sources.append(candidate)

print(f"discovered {len(sources)} source file(s):")
for p in sources:
    print(f"  - {p.relative_to(FIXTURES)}")

## 4. Analyse + flatten into a `DataFrame`

Per-file `bca.analyze(path)` is used here (not `bca.analyze_batch`) because the single-file entry point honours the CLI walker's `@generated` / `DO NOT EDIT` filter — `analyze_batch` hardcodes `skip_generated=False` and would otherwise pull generated code into the chart. `flatten_spaces` walks each result's `FuncSpace` tree and yields one row per function / class / namespace. The whole table fits into a `pandas.DataFrame` with a single `pd.DataFrame.from_records` call.

In [ ]:
rows: list[dict[str, object]] = []
skipped: list[Path] = []
errors: list[tuple[Path, Exception]] = []
for path in sources:
    try:
        result = bca.analyze(path)
    except (OSError, ValueError) as exc:
        errors.append((path, exc))
        continue
    if result is None:
        skipped.append(path)
        continue
    # Anchor each flattened row to its source file so per-path
    # joins / group-bys work downstream.
    rows.extend({"source": path.name, **record} for record in bca.flatten_spaces(result))

print(
    f"analysed {len(sources) - len(skipped) - len(errors)} files, "
    f"skipped {len(skipped)} generated, {len(errors)} errors"
)
df = pd.DataFrame.from_records(rows)
df.head()

## 5. Plot cyclomatic complexity per function

Filter to function-kind rows (file-level `unit` rows aggregate across the whole file and would dominate the chart) and plot one bar per function.

In [ ]:
functions = df[df["kind"] == "function"].copy()
functions["label"] = functions["source"] + ":" + functions["name"].astype(str)

fig, ax = plt.subplots(figsize=(8, 4))
functions.plot.bar(x="label", y="cyclomatic.sum", ax=ax, legend=False)
ax.set_ylabel("cyclomatic.sum")
ax.set_xlabel("function")
ax.set_title("Cyclomatic complexity per function")
plt.tight_layout()
plt.show()

## 6. Top-N most complex functions

The same DataFrame is the natural input to a `.nlargest(...)` lookup. Adjust `TOP_N` to taste — the fixture set is small, so 3 is plenty.

In [ ]:
TOP_N = 3
top = functions.nlargest(TOP_N, "cyclomatic.sum")[
    ["source", "name", "start_line", "end_line", "cyclomatic.sum"]
]
top

## Where to go next

* [`examples/cli_parity.py`](cli_parity.py) — assert byte-for-byte parity between `bca.analyze` and `bca metrics --output-format json`.
* [`examples/pipeline_db.py`](pipeline_db.py) — the same flow as this notebook, persisted to sqlite, with a deliberately-broken file in the batch.
* [`examples/sarif_upload.py`](sarif_upload.py) — produce SARIF 2.1.0 output ready for GitHub Code Scanning.
* The book's [Python Bindings](../../big-code-analysis-book/src/python/README.md) chapter covers the full API surface page by page.